# Module-05: Guided Lab

In [ ]:
# Install the "hmmlearn" library.
# This library is used to work with Hidden Markov Models (HMMs),
# which are useful for sequence data like time series or text.
!pip install hmmlearn

# Install the "tensorflow" library.
# TensorFlow is a deep learning framework used to build and train neural networks.
!pip install tensorflow

# Install the "keras" library.
# Keras is a high-level API that runs on top of TensorFlow.
# It makes it easier to define and train deep learning models.
!pip install keras


This code demonstrates Generating N-grams in Python using NLTK. It takes a sample sentence, tokenizes it into individual words, and then uses a helper function to create unigrams (single words), bigrams (pairs of words), and trigrams (triples of words). Finally, it prints each list so you can see how the text is broken into different-sized word sequences.

In [ ]:
# Import the ngrams function from NLTK.
# This helps us create sequences of N words (like pairs or triples of words).
from nltk import ngrams

# Import Counter (not used in this example, but often helpful for counting N-grams)
from collections import Counter

# Import the main NLTK module so we can use its functions
import nltk

# Download the "punkt" tokenizer data (only needed the first time you run this code).
# This allows NLTK to split text into words and sentences.
nltk.download('punkt')

# Sometimes "punkt_tab" is also needed in certain environments.
nltk.download('punkt_tab')

# ---------------------------------------
# 1. SAMPLE TEXT
# ---------------------------------------

# Our example sentence stored in a variable called "text".
text = "Natural Language Processing is a fascinating field of study."

# ---------------------------------------
# 2. TOKENIZE THE TEXT INTO WORDS
# ---------------------------------------

# Use NLTK's word_tokenize to split the sentence into a list of words.
# Example result: ["Natural", "Language", "Processing", "is", ...]
tokens = nltk.word_tokenize(text)

# ---------------------------------------
# 3. FUNCTION TO GENERATE N-GRAMS
# ---------------------------------------

def generate_ngrams(tokens, n):
    """
    Given a list of tokens (words) and a number n,
    return a list of N-grams as strings.

    For example:
    - n = 1 -> unigrams (single words)
    - n = 2 -> bigrams (pairs of words)
    - n = 3 -> trigrams (triples of words)
    """
    # ngrams(tokens, n) creates an iterable of tuples, each with n words.
    # Example for bigrams: [("Natural", "Language"), ("Language", "Processing"), ...]
    n_grams = ngrams(tokens, n)

    # Join each tuple of words into a string separated by spaces.
    # Example: ("Natural", "Language") -> "Natural Language"
    return [' '.join(grams) for grams in n_grams]

# ---------------------------------------
# 4. GENERATE UNIGRAMS, BIGRAMS, AND TRIGRAMS
# ---------------------------------------

# Unigrams: sequences of 1 word
unigrams = generate_ngrams(tokens, 1)

# Bigrams: sequences of 2 words
bigrams = generate_ngrams(tokens, 2)

# Trigrams: sequences of 3 words
trigrams = generate_ngrams(tokens, 3)

# ---------------------------------------
# 5. PRINT THE RESULTS
# ---------------------------------------

print("Unigrams:")
print(unigrams)

print("\nBigrams:")
print(bigrams)

print("\nTrigrams:")
print(trigrams)


This code shows Training an N-gram Language Model by building a simple bigram (2-gram) model from a small text corpus. It tokenizes each sentence into words, counts how often each word pair (w1, w2) appears, then converts those counts into probabilities P(w2 | w1). Finally, it lets you query the model for the probability of seeing one word after another, such as the probability of “NLP” following “for.”

In [ ]:
# Import defaultdict from the collections module.
# defaultdict lets us create dictionaries that automatically give a default value
# when a new key is accessed (so we don't get KeyError).
from collections import defaultdict

# NOTE: This code also uses nltk.word_tokenize and ngrams.
# Make sure you have already done:
#   import nltk
#   from nltk import ngrams
#   nltk.download('punkt')

# ---------------------------------------
# 1. SAMPLE TEXT CORPUS
# ---------------------------------------

# A small list of example sentences about NLP and machine learning.
corpus = [
    "Natural Language Processing is a fascinating field of study.",
    "Machine learning and NLP are closely related.",
    "Language models are essential for NLP tasks."
]

# ---------------------------------------
# 2. TOKENIZE THE CORPUS INTO WORDS
# ---------------------------------------

# For each sentence in the corpus, split it into a list of words (tokens).
# tokenized_corpus will be a list of lists:
# [
#   ["Natural", "Language", "Processing", "is", ...],
#   ["Machine", "learning", "and", "NLP", ...],
#   ...
# ]
tokenized_corpus = [nltk.word_tokenize(sentence) for sentence in corpus]

# ---------------------------------------
# 3. TRAIN BIGRAM LANGUAGE MODEL
# ---------------------------------------

def train_bigram_model(tokenized_corpus):
    """
    Build a simple bigram model from the tokenized corpus.
    A bigram is a pair of consecutive words (w1, w2).
    The model will store P(w2 | w1) = probability of w2 given w1.
    """

    # model[w1][w2] will store the probability of seeing w2 after w1.
    # We use defaultdict so that missing entries start at 0 automatically.
    model = defaultdict(lambda: defaultdict(lambda: 0))

    # ----- Step 1: Count bigram frequencies -----
    for sentence in tokenized_corpus:
        # ngrams(sentence, 2) gives all bigrams (w1, w2) in the sentence.
        for w1, w2 in ngrams(sentence, 2):
            model[w1][w2] += 1  # increment the count for this bigram

    # ----- Step 2: Convert counts to probabilities -----
    for w1 in model:
        # total_count = total number of times w1 is followed by any word
        total_count = float(sum(model[w1].values()))
        for w2 in model[w1]:
            # Divide each bigram count by total_count to get a probability
            model[w1][w2] /= total_count

    return model

# Train the bigram model using our tokenized corpus
bigram_model = train_bigram_model(tokenized_corpus)

# ---------------------------------------
# 4. FUNCTION TO LOOK UP BIGRAM PROBABILITY
# ---------------------------------------

def get_bigram_probability(bigram_model, w1, w2):
    """
    Return the probability of seeing word w2 after word w1,
    based on the trained bigram model.
    If the pair (w1, w2) never appeared, this will be 0.
    """
    return bigram_model[w1][w2]

# ---------------------------------------
# 5. TEST THE MODEL
# ---------------------------------------

print("Bigram Probability (NLP | for):")
# This means: P('NLP' | 'for') = probability of 'NLP' coming after 'for'
print(get_bigram_probability(bigram_model, 'for', 'NLP'))


This code demonstrates Implementing HMMs in Python using the hmmlearn library on a tiny part-of-speech-style example. It defines two hidden states (Noun and Verb), a small vocabulary of observed words, and manually sets the start, transition, and emission probability matrices for the Hidden Markov Model. A short sentence ("I run to the store") is encoded as one-hot vectors and passed into the model, which then uses the Viterbi algorithm to decode the most likely sequence of hidden states. Finally, it prints the original words along with their predicted hidden states (e.g., whether each word is treated as a noun or a verb).

In [ ]:
# Import NumPy for working with arrays and matrices
import numpy as np

# Import the Hidden Markov Model (HMM) implementation from hmmlearn
from hmmlearn import hmm

# ---------------------------------------
# 1. DEFINE STATES AND OBSERVATIONS
# ---------------------------------------

# Hidden states in our model.
# We pretend each word in the sentence is either a Noun or a Verb.
states = ["Noun", "Verb"]
n_states = len(states)  # Number of hidden states

# Possible observed words in our tiny vocabulary
observations = ["I", "run", "to", "the", "store"]
n_observations = len(observations)  # Number of different observed words

# ---------------------------------------
# 2. TRANSITION PROBABILITY MATRIX (A)
# ---------------------------------------

# transition_probability[i][j] = probability of going from state i to state j
# Row 0 is from Noun, row 1 is from Verb
transition_probability = np.array([
    [0.7, 0.3],  # From Noun to [Noun, Verb]
    [0.4, 0.6]   # From Verb to [Noun, Verb]
])

# ---------------------------------------
# 3. EMISSION PROBABILITY MATRIX (B)
# ---------------------------------------

# emission_probability[i][j] = probability of emitting observation j from state i
# Row 0 is for state Noun, row 1 is for state Verb
# Columns correspond to: ["I", "run", "to", "the", "store"]
emission_probability = np.array([
    [0.2, 0.3, 0.2, 0.1, 0.2],  # Probabilities of each word if the hidden state is Noun
    [0.1, 0.6, 0.1, 0.1, 0.1]   # Probabilities of each word if the hidden state is Verb
])

# ---------------------------------------
# 4. INITIAL STATE PROBABILITIES (pi)
# ---------------------------------------

# start_probability[i] = probability that the first hidden state is i
# Here we start in Noun with probability 0.6 and Verb with probability 0.4
start_probability = np.array([0.6, 0.4])  # [Noun, Verb]

# ---------------------------------------
# 5. CREATE AND CONFIGURE THE HMM MODEL
# ---------------------------------------

# Create a Multinomial HMM model.
# n_components = number of hidden states
# n_trials is used for the total number of trials for each observed vector.
# Since we are essentially providing one-hot encoded observations (one trial per step),
# n_trials should be 1 for each observation. The model handles this internally
# if the input X is appropriately shaped for multinomial counts.
model = hmm.MultinomialHMM(n_components=n_states, n_trials=1)

# Set the parameters of the HMM model:
# - starting state probabilities
# - transition probabilities
# - emission probabilities
model.startprob_ = start_probability
model.transmat_ = transition_probability
model.emissionprob_ = emission_probability

# ---------------------------------------
# 6. ENCODE OBSERVATIONS AS NUMBERS
# ---------------------------------------

# We need to represent observations as integers for the HMM.
# "I"    -> 0
# "run"  -> 1
# "to"   -> 2
# "the"  -> 3
# "store"-> 4
raw_observation_sequence = [0, 1, 2, 3, 4]  # sequence: "I run to the store"

# Convert to a one-hot encoded NumPy array as required by the current MultinomialHMM.
# Each row represents a time step, and columns represent observation counts.
# For a single observation, it will be a 1 at the corresponding index and 0s elsewhere.
observation_sequence = np.zeros((len(raw_observation_sequence), n_observations), dtype=int)
for i, obs_idx in enumerate(raw_observation_sequence):
    observation_sequence[i, obs_idx] = 1

# ---------------------------------------
# 7. DECODE HIDDEN STATES WITH VITERBI
# ---------------------------------------

# model.decode finds the most likely sequence of hidden states
# given the observation sequence.
# We use the "viterbi" algorithm which is a standard decoding method for HMMs.
logprob, hidden_states = model.decode(observation_sequence, algorithm="viterbi")

# ---------------------------------------
# 8. PRINT RESULTS
# ---------------------------------------

# Convert numeric observations back to their word form
decoded_observations = [observations[i] for i in raw_observation_sequence]

# Convert numeric hidden states back to "Noun" or "Verb"
decoded_states = [states[i] for i in hidden_states]

print("Observations:", decoded_observations)
print("Hidden states:", decoded_states)


https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340


Observations: ['I', 'run', 'to', 'the', 'store']
Hidden states: ['Noun', 'Noun', 'Noun', 'Noun', 'Noun']


This code demonstrates how to train a Hidden Markov Model (HMM) using hmmlearn as part of Solving the Three Fundamental Problems of HMMs. It starts with several sequences of observations (words encoded as integers), reshapes and concatenates them into the format expected by hmmlearn, then fits a MultinomialHMM to automatically learn the start, transition, and emission probabilities from the data. After training, it prints these learned parameters, which are essential for later tackling the evaluation and decoding problems (computing sequence likelihoods and finding the most likely hidden state sequence).

In [ ]:
# ---------------------------------------
# 1. TRAINING SEQUENCES (OBSERVATION DATA)
# ---------------------------------------

# Sample data: each list is a sequence of observed words, encoded as integers.
# For example, you might have the mapping:
#   0 -> "I"
#   1 -> "run"
#   2 -> "to"
#   3 -> "the"
#   4 -> "store"
#
# These three sequences are different word orders using the same vocabulary.
training_sequences = [
    [0, 1, 2, 3, 4],  # "I run to the store"
    [4, 2, 0, 1, 3],  # "store to I run the"
    [1, 2, 3, 0, 4],  # "run to the I store"
]

# ---------------------------------------
# 2. PREPARE DATA FOR hmmlearn
# ---------------------------------------

# hmmlearn expects:
#  - one long 2D array of all observations
#  - a list of lengths, telling it where each sequence ends
#
# Step 1: reshape each sequence into a 2D array of shape (sequence_length, 1)
# because hmmlearn expects observations as column vectors.
training_sequences = [np.array(seq).reshape(-1, 1) for seq in training_sequences]

# Step 2: store the length of each sequence.
# This helps the model know how to split the long array back into sequences.
lengths = [len(seq) for seq in training_sequences]

# Step 3: concatenate all sequences into one long array of observations.
training_data = np.concatenate(training_sequences)

# ---------------------------------------
# 3. CREATE AND TRAIN THE HMM MODEL
# ---------------------------------------

# Create a Multinomial Hidden Markov Model (HMM).
# - n_components = number of hidden states (for example, Noun and Verb)
# - n_iter = maximum number of training iterations
# NOTE: n_states should be defined earlier (e.g., n_states = 2 for ["Noun", "Verb"])
model = hmm.MultinomialHMM(n_components=n_states, n_iter=100)

# Train (fit) the HMM on the training data.
# - training_data: all observations stacked together
# - lengths: tells the model where each sequence ends
model.fit(training_data, lengths)

# ---------------------------------------
# 4. INSPECT THE LEARNED PARAMETERS
# ---------------------------------------

print("Learned start probabilities:")
# startprob_ is the learned probability of starting in each hidden state.
print(model.startprob_)

print("\nLearned transition probabilities:")
# transmat_ is the learned probability of moving from one state to another.
print(model.transmat_)

print("\nLearned emission probabilities:")
# emissionprob_ is the learned probability of each observation given a state.
print(model.emissionprob_)


This code demonstrates Implementing RNNs in Python with TensorFlow/Keras by training a simple character-level Recurrent Neural Network (RNN) to predict the next character in a short text sequence. The program starts with the phrase “hello world”, builds a character vocabulary, and creates training pairs where each input is a 3-character sequence and the label is the next character. The data is reshaped into the format required by RNNs, and a SimpleRNN model is trained to learn these patterns. After training, the model can generate new text by repeatedly predicting the next character based on the previous ones, showing how RNNs can learn and extend character-level patterns.


In [ ]:
# Import NumPy for working with numeric arrays
import numpy as np

# Import TensorFlow and Keras tools for building neural networks
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN
from tensorflow.keras.utils import to_categorical

# ---------------------------------------
# 1. SAMPLE TEXT CORPUS
# ---------------------------------------

# Our very small example text. The model will learn patterns in these characters.
text = "hello world"

# ---------------------------------------
# 2. CREATE CHARACTER LEVEL VOCABULARY
# ---------------------------------------

# Get a sorted list of all unique characters in the text.
# Example: [' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w']
chars = sorted(set(text))

# Create a dictionary that maps each character to a unique index.
# Example: {' ': 0, 'd': 1, 'e': 2, ...}
char_to_idx = {char: idx for idx, char in enumerate(chars)}

# Create the reverse mapping, from index to character.
idx_to_char = {idx: char for char, idx in char_to_idx.items()}

# ---------------------------------------
# 3. CREATE INPUT OUTPUT PAIRS FOR TRAINING
# ---------------------------------------

# We will use sequences of 3 characters to predict the next character.
sequence_length = 3

X = []  # list for input sequences
y = []  # list for target characters (the next character)

# Loop over the text and create sliding windows of length 3.
# For each window of 3 characters, the target is the next character.
for i in range(len(text) - sequence_length):
    # Take a sequence of 3 characters and convert them to indices.
    X.append([char_to_idx[char] for char in text[i:i + sequence_length]])
    # Take the next character after the sequence and convert to index.
    y.append(char_to_idx[text[i + sequence_length]])

# Convert lists to NumPy arrays.
X = np.array(X)

# Convert y to one hot encoded vectors.
# to_categorical turns each index into a vector with a 1 at that index and 0 elsewhere.
y = to_categorical(y, num_classes=len(chars))

# ---------------------------------------
# 4. RESHAPE INPUT FOR RNN
# ---------------------------------------

# RNNs expect input in the shape:
# (number_of_sequences, sequence_length, number_of_features)
# Here, number_of_features is 1, since each time step has a single integer value.
X = X.reshape((X.shape[0], X.shape[1], 1))

# ---------------------------------------
# 5. DEFINE THE RNN MODEL
# ---------------------------------------

# Use a Sequential model, which lets us stack layers one after another.
model = Sequential()

# Add a SimpleRNN layer with 50 units.
# input_shape tells the layer the size of each input:
# - sequence_length time steps
# - 1 feature per time step
model.add(SimpleRNN(50, input_shape=(sequence_length, 1)))

# Add a Dense output layer with softmax activation.
# The number of units is equal to the number of unique characters.
# softmax outputs a probability distribution over all possible characters.
model.add(Dense(len(chars), activation='softmax'))

# ---------------------------------------
# 6. COMPILE THE MODEL
# ---------------------------------------

# Compile the model by choosing:
# - optimizer: "adam" for efficient training
# - loss: "categorical_crossentropy" for multi class classification
model.compile(optimizer='adam', loss='categorical_crossentropy')

# ---------------------------------------
# 7. TRAIN THE MODEL
# ---------------------------------------

# Train the model on our data.
# epochs is how many times we go through the entire training set.
model.fit(X, y, epochs=100, verbose=1)

# ---------------------------------------
# 8. FUNCTION TO GENERATE TEXT
# ---------------------------------------

def generate_text(model, start_string, num_generate):
    """
    Generate new text using the trained model.

    start_string: initial characters to begin with
    num_generate: how many new characters to generate
    """

    # Convert the starting string into indices using our mapping.
    input_eval = [char_to_idx[s] for s in start_string]

    # Convert to a NumPy array and reshape to match RNN input format:
    # (batch_size=1, sequence_length, features=1)
    input_eval = np.array(input_eval).reshape((1, len(input_eval), 1))

    # List to store generated characters
    text_generated = []

    # Generate characters one by one
    for i in range(num_generate):
        # Get model predictions for the current input sequence.
        # predictions has shape (1, number_of_characters).
        predictions = model.predict(input_eval)

        # Choose the index with the highest predicted probability.
        predicted_id = np.argmax(predictions[-1])

        # Reshape predicted_id so we can append it to the existing sequence.
        predicted_id_reshaped = np.array(predicted_id).reshape((1, 1, 1))

        # Slide the window forward by 1:
        # drop the first time step and add the new predicted character at the end.
        input_eval = np.append(input_eval[:, 1:], predicted_id_reshaped, axis=1)

        # Convert the predicted index back to a character and store it.
        text_generated.append(idx_to_char[predicted_id])

    # Return the starting string plus all the generated characters.
    return start_string + ''.join(text_generated)

# ---------------------------------------
# 9. USE THE MODEL TO GENERATE NEW TEXT
# ---------------------------------------

# Starting seed text. This should be length 3, since sequence_length = 3.
start_string = "hel"

# Generate 5 new characters based on the seed.
generated_text = generate_text(model, start_string, 5)

print("Generated text:")
print(generated_text)



This code demonstrates how to build and train a character-level text generator using an LSTM network — an improved version of a basic RNN that can remember longer patterns in sequences. The program creates a vocabulary of characters from the phrase "hello world", turns short sequences of characters into training examples, and trains an LSTM model using TensorFlow/Keras to predict the next character. After training, the model can generate new text by repeatedly predicting the next character based on the previous ones. This example shows how LSTMs improve traditional RNNs by handling longer dependencies and producing more accurate sequence predictions.

In [ ]:
# Import tools from Keras (part of TensorFlow) to build neural networks.
# Sequential: lets us stack layers in order.
# LSTM: a type of RNN that can remember longer patterns in sequences.
# Dense: a fully connected (output) layer.
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Import NumPy for working with numeric arrays.
import numpy as np

# Import TensorFlow (backend for Keras).
import tensorflow as tf

# (These imports are not needed for this LSTM example, but shown here for reference.)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN
from tensorflow.keras.utils import to_categorical

# ---------------------------------------
# 1. SAMPLE TEXT CORPUS
# ---------------------------------------

# Our small example text. The model will learn character patterns from this string.
text = "hello world"

# ---------------------------------------
# 2. CREATE CHARACTER-LEVEL VOCABULARY
# ---------------------------------------

# Get all unique characters in the text and sort them.
# Example: [' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w']
chars = sorted(set(text))

# Map each character to a unique index (number).
# Example: {' ': 0, 'd': 1, 'e': 2, ...}
char_to_idx = {char: idx for idx, char in enumerate(chars)}

# Create the reverse mapping: from index back to character.
idx_to_char = {idx: char for char, idx in char_to_idx.items()}

# ---------------------------------------
# 3. CREATE INPUT–OUTPUT PAIRS FOR TRAINING
# ---------------------------------------

# We will use sequences of 3 characters to predict the next character.
sequence_length = 3

X = []  # will hold input sequences (as indices)
y = []  # will hold the target next character (as index)

# Slide a window of length 3 over the text.
# For each window, the model will try to predict the next character.
for i in range(len(text) - sequence_length):
    # Take 3 characters and convert them to their indices.
    X.append([char_to_idx[char] for char in text[i:i + sequence_length]])
    # The label is the character right after the 3-character window.
    y.append(char_to_idx[text[i + sequence_length]])

# Convert lists to NumPy arrays so they can be used by TensorFlow.
X = np.array(X)

# Convert y (a list of indices) into one-hot encoded vectors.
# Example: if there are 8 unique characters, each label becomes a vector of length 8.
y = to_categorical(y, num_classes=len(chars))

# ---------------------------------------
# 4. RESHAPE INPUT FOR LSTM
# ---------------------------------------

# LSTM layers expect input in this shape:
# (number_of_sequences, sequence_length, number_of_features)
# number_of_features = 1, because each time step is a single integer (character index).
X = X.reshape((X.shape[0], X.shape[1], 1))

# ---------------------------------------
# 5. DEFINE THE LSTM MODEL
# ---------------------------------------

# Create a Sequential model (a stack of layers).
model = Sequential()

# Add an LSTM layer with 50 units.
# input_shape tells the layer:
#   - sequence_length: how many time steps (3 characters)
#   - 1: number of features per time step
model.add(LSTM(50, input_shape=(sequence_length, 1)))

# Add a Dense output layer with softmax activation.
# The number of units equals the number of unique characters.
# softmax gives a probability for each possible next character.
model.add(Dense(len(chars), activation='softmax'))

# ---------------------------------------
# 6. COMPILE THE MODEL
# ---------------------------------------

# Compile the model by specifying:
# - optimizer: "adam" (a good default for many problems)
# - loss function: "categorical_crossentropy" for multi-class classification
model.compile(optimizer='adam', loss='categorical_crossentropy')

# ---------------------------------------
# 7. TRAIN THE MODEL
# ---------------------------------------

# Train the model on our input (X) and labels (y).
# epochs=20 means the model sees the whole dataset 20 times.
# validation_data=(X, y) just shows how well the model does on the same data
# during training (for monitoring).
model.fit(X, y, epochs=20, validation_data=(X, y))

# ---------------------------------------
# 8. FUNCTION TO GENERATE TEXT
# ---------------------------------------

def generate_text(model, start_string, num_generate):
    """
    Generate new characters using the trained LSTM model.

    start_string: the initial characters we start with (must match sequence_length).
    num_generate: how many new characters to generate.
    """

    # Convert the starting characters into indices using our mapping.
    input_eval = [char_to_idx[s] for s in start_string]

    # Reshape into the format the model expects:
    # (batch_size=1, sequence_length, number_of_features=1)
    input_eval = np.array(input_eval).reshape((1, len(input_eval), 1))

    # List to store the generated characters.
    text_generated = []

    # Generate characters one at a time.
    for i in range(num_generate):
        # Get the model's prediction for the next character.
        # predictions shape: (1, number_of_characters)
        predictions = model.predict(input_eval)

        # Choose the character with the highest predicted probability.
        predicted_id = np.argmax(predictions[-1])

        # Reshape predicted_id to have the correct shape for appending to input_eval:
        # (batch_size=1, time_steps=1, features=1)
        predicted_id_reshaped = np.array(predicted_id).reshape((1, 1, 1))

        # Slide the input window forward by 1 time step:
        # - drop the first character
        # - append the newly predicted character at the end
        input_eval = np.append(input_eval[:, 1:], predicted_id_reshaped, axis=1)

        # Convert the predicted index back to a character and store it.
        text_generated.append(idx_to_char[predicted_id])

    # Return the original start_string plus the newly generated characters.
    return start_string + ''.join(text_generated)

# ---------------------------------------
# 9. USE THE MODEL TO GENERATE NEW TEXT
# ---------------------------------------

# Starting text for generation.
# It must be 3 characters long to match sequence_length.
start_string = "hel"

# Ask the model to generate 5 new characters after "hel".
generated_text = generate_text(model, start_string, 5)

print("Generated text:")
print(generated_text)


This code demonstrates Implementing LSTMs in Python with TensorFlow/Keras to build a simple character-level text generator. It takes a small text string (“hello world”) and trains an LSTM model to predict the next character based on a sequence of three characters. The code creates a character vocabulary, prepares input–output training pairs, reshapes the data for LSTM input, and trains a neural network to learn character patterns. After training, the model can generate new text by predicting characters one at a time, starting from a given input like "hel".

In [ ]:
# Import NumPy for working with numeric arrays
import numpy as np

# Import TensorFlow and Keras tools for building neural networks
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
from tensorflow.keras.utils import to_categorical

# ---------------------------------------
# 1. SAMPLE TEXT CORPUS
# ---------------------------------------

# Our tiny example text. The model will learn patterns between characters in this string.
text = "hello world"

# ---------------------------------------
# 2. CREATE A CHARACTER-LEVEL VOCABULARY
# ---------------------------------------

# Get all unique characters in the text and sort them.
# Example result: [' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w']
chars = sorted(set(text))

# Create a dictionary that maps each character to a unique integer index.
# For example: {' ': 0, 'd': 1, 'e': 2, ...}
char_to_idx = {char: idx for idx, char in enumerate(chars)}

# Create the reverse mapping: from index back to character.
# For example: {0: ' ', 1: 'd', 2: 'e', ...}
idx_to_char = {idx: char for char, idx in char_to_idx.items()}

# ---------------------------------------
# 3. CREATE INPUT–OUTPUT PAIRS FOR TRAINING
# ---------------------------------------

# We will teach the model to predict the next character
# given a sequence of 3 characters.
sequence_length = 3

X = []  # list to store input sequences (as indices)
y = []  # list to store the next character (as index)

# Slide a window of length 3 across the text
for i in range(len(text) - sequence_length):
    # Take 3 characters and convert each to its index
    X.append([char_to_idx[char] for char in text[i:i + sequence_length]])
    # The label is the character that comes right after this 3-char sequence
    y.append(char_to_idx[text[i + sequence_length]])

# Convert X to a NumPy array so it can be used as model input
X = np.array(X)

# Convert y (list of integer labels) into one-hot encoded vectors
# If there are N unique characters, each label becomes a vector of length N
y = to_categorical(y, num_classes=len(chars))

# ---------------------------------------
# 4. RESHAPE INPUT FOR LSTM
# ---------------------------------------

# LSTM expects input in the shape:
# (number_of_sequences, sequence_length, number_of_features)
# Here each time step has 1 feature (the character index), so features = 1
X = X.reshape((X.shape[0], X.shape[1], 1))

# ---------------------------------------
# 5. DEFINE THE LSTM MODEL
# ---------------------------------------

# Use a Sequential model, which is a simple stack of layers.
model = Sequential()

# Add an LSTM layer with 50 units.
# input_shape = (sequence_length, 1):
#   - sequence_length: how many time steps (3 characters)
#   - 1: number of features per time step
model.add(LSTM(50, input_shape=(sequence_length, 1)))

# Add a Dense output layer with softmax activation.
# The number of units equals the number of unique characters.
# softmax outputs a probability distribution over all characters.
model.add(Dense(len(chars), activation='softmax'))

# ---------------------------------------
# 6. COMPILE THE MODEL
# ---------------------------------------

# Compile the model by choosing:
# - optimizer: "adam" (a common choice that works well in many cases)
# - loss: "categorical_crossentropy" (used for multi-class classification)
model.compile(optimizer='adam', loss='categorical_crossentropy')

# ---------------------------------------
# 7. TRAIN THE MODEL
# ---------------------------------------

# Train the model on our data (X as input, y as labels).
# epochs=100 means the model will see the entire dataset 100 times.
model.fit(X, y, epochs=100, verbose=1)

# ---------------------------------------
# 8. FUNCTION TO GENERATE TEXT
# ---------------------------------------

def generate_text(model, start_string, num_generate):
    """
    Generate new text characters using the trained LSTM model.

    start_string: the starting sequence of characters (length should be 3 here)
    num_generate: how many new characters to generate
    """

    # Convert the starting characters into their index form
    input_eval = [char_to_idx[s] for s in start_string]

    # Reshape to the format expected by the LSTM:
    # (batch_size=1, sequence_length, features=1)
    input_eval = np.array(input_eval).reshape((1, len(input_eval), 1))

    # List to store generated characters
    text_generated = []

    # Generate characters one by one
    for i in range(num_generate):
        # Get the model's probability predictions for the next character
        predictions = model.predict(input_eval)

        # Choose the index of the character with the highest probability
        predicted_id = np.argmax(predictions[-1])

        # Reshape predicted_id to match the input format:
        # (batch_size=1, time_steps=1, features=1)
        predicted_id_reshaped = np.array(predicted_id).reshape((1, 1, 1))

        # Slide the input window:
        # - remove the first time step
        # - append the newly predicted character at the end
        input_eval = np.append(input_eval[:, 1:], predicted_id_reshaped, axis=1)

        # Convert predicted index back to a character and add to the result
        text_generated.append(idx_to_char[predicted_id])

    # Return the original start_string plus all newly generated characters
    return start_string + ''.join(text_generated)

# ---------------------------------------
# 9. USE THE MODEL TO GENERATE NEW TEXT
# ---------------------------------------

# Starting sequence (must have length 3, since sequence_length = 3)
start_string = "hel"

# Generate 5 new characters after the starting sequence
generated_text = generate_text(model, start_string, 5)

print("Generated text:")
print(generated_text)
